# Módulos y Patrones de Estructuración en Rust

Este notebook explica cómo organizar el código en Rust utilizando su sistema de módulos y cómo estructurar objetos utilizando patrones comunes de diseño.

## 1. El Sistema de Módulos (Crates, Modules y Paths)

Rust utiliza un sistema de módulos jerárquico para controlar el alcance (scope) y la privacidad.

- **Crate**: Un paquete de Rust (binario o librería).
- **Module**: Organiza el código dentro de un crate.
- **Path**: Forma de nombrar un ítem (ej. `std::collections::HashMap`).

### Privacidad por defecto
En Rust, todo es **privado** por defecto. Para que algo sea accesible desde fuera de su módulo, debe marcarse con la palabra clave `pub`.

In [ ]:
mod comunicacion {
    // Este módulo es privado para el exterior, pero visible para su padre
    pub mod interno {
        pub fn hola() {
            println!("Hola desde el módulo interno!");
        }

        fn secreto() {
            println!("Esto es privado");
        }
    }

    pub fn llamar_hola() {
        // Dentro del mismo módulo o descendientes podemos llamar a lo público
        interno::hola();
    }
}

fn main() {
    // Uso de rutas (paths)
    comunicacion::llamar_hola();
    comunicacion::interno::hola();
}

## 2. Estructura de Archivos

Normalmente no definimos todos los módulos en el mismo archivo. Rust busca los módulos basándose en el nombre:

1. `mod jardin;` en `main.rs` busca el código en `src/jardin.rs` o `src/jardin/mod.rs` (estilo antiguo).
2. Submódulos: `mod flores;` dentro de `jardin.rs` busca en `src/jardin/flores.rs`.

### Re-exportación (`pub use`)
A veces la estructura interna es compleja pero queremos ofrecer una API sencilla. Usamos `pub use` para traer ítems internos al nivel superior.

In [ ]:
mod capa_interna {
    pub struct Motor {
        pub potencia: u32,
    }
}

// Re-exportación
pub use capa_interna::Motor;

fn main() {
    // El usuario no necesita saber que Motor estaba en capa_interna
    let m = Motor { potencia: 150 };
}

## 3. Patrones de Estructuración de Objetos

Rust no tiene clases tradicionales (herencia), por lo que usa patrones basados en **Composición** y **Traits**.

### A. El Patrón Constructor (`new`)
Por convención, se usa una función asociada llamada `new` para inicializar estructuras.

In [ ]:
pub struct Config {
    pub puerto: u16,
    host: String, // Campo privado
}

impl Config {
    pub fn new(host: &str) -> Self {
        Config {
            puerto: 8080,
            host: host.to_string(),
        }
    }
}

### B. El Patrón Builder
Útil cuando un objeto tiene muchos parámetros opcionales o una configuración compleja.

In [ ]:
pub struct Servidor {
    host: String,
    puerto: u16,
    timeout: u32,
}

pub struct ServidorBuilder {
    host: String,
    puerto: Option<u16>,
    timeout: Option<u32>,
}

impl ServidorBuilder {
    pub fn new(host: &str) -> Self {
        ServidorBuilder {
            host: host.to_string(),
            puerto: None,
            timeout: None,
        }
    }

    pub fn puerto(mut self, p: u16) -> Self {
        self.puerto = Some(p);
        self
    }

    pub fn build(self) -> Servidor {
        Servidor {
            host: self.host,
            puerto: self.puerto.unwrap_or(80),
            timeout: self.timeout.unwrap_or(30),
        }
    }
}

// Uso:
// let srv = ServidorBuilder::new("localhost").puerto(3000).build();

### C. Newtype Pattern
Se usa para encapsular un tipo existente y añadirle seguridad o implementar traits externos. Es excelente para la seguridad de tipos (ej. no mezclar IDs de usuarios con IDs de productos).

In [ ]:
struct Metros(u32);
struct Pies(u32);

fn calcular_distancia(m: Metros) {
    println!("Distancia en metros: {}", m.0);
}

let d = Metros(100);
// calcular_distancia(Pies(100)); // ERROR: Tipos diferentes

### D. Estados con Enums (State Pattern)
En lugar de usar herencia, usamos Enums para representar estados que contienen diferentes datos.

In [ ]:
enum EstadoPedido {
    Procesando,
    Enviado(String), // Contiene código de seguimiento
    Entregado { fecha: String },
}

fn revisar_pedido(p: EstadoPedido) {
    match p {
        EstadoPedido::Procesando => println!("Aún en almacen"),
        EstadoPedido::Enviado(tracking) => println!("En camino: {}", tracking),
        EstadoPedido::Entregado { fecha } => println!("Llegó el {}", fecha),
    }
}

## 4. Estructuración por Responsabilidad

Un patrón común en aplicaciones Rust es dividir el código en:

1. **`models.rs`**: Definición de `structs` y `enums`.
2. **`logic.rs`**: Implementaciones de lógica de negocio.
3. **`api.rs` o `ui.rs`**: Interfaces externas.

### El uso de `self`, `super` y `crate` en rutas
- `self`: El módulo actual.
- `super`: El módulo padre.
- `crate`: La raíz del crate actual.